SETUP

In [51]:
import tensorflow as tf
import cv2
import numpy as np
import time

MODEL_PATH = "hrnet_pose.tflite"
DELEGATE_PATH = "libQnnTFLiteDelegate.so"

def load_interpreter(model_path, use_npu=True):
    if use_npu:
        try:
            delegate = tf.lite.experimental.load_delegate(DELEGATE_PATH, options={"backend_type": "htp"})
            interp = tf.lite.Interpreter(model_path=model_path, experimental_delegates=[delegate])
            interp.allocate_tensors()
            print("Loaded QNN HTP (NPU) delegate.")
            return interp, "npu"
        except (ValueError, OSError) as e:
            print(f"Could not load NPU delegate ({e}). Falling back to CPU.")

    interp = tf.lite.Interpreter(model_path=model_path)
    interp.allocate_tensors()
    return interp, "cpu"

interpreter_npu, backend = load_interpreter("hrnet_pose.tflite", use_npu=True)
input_details = interpreter_npu.get_input_details()
output_details = interpreter_npu.get_output_details()
scale, zero_point = output_details[0]["quantization"]

IN_H, IN_W = input_details[0]["shape"][1], input_details[0]["shape"][2]

print(f"Active backend: {backend}")
print(f"Input size (H,W): ({IN_H}, {IN_W})")
print(f"Output quantization: scale={scale}, zero_point={zero_point}")


Starting stage: Graph Preparation Initializing
Loaded QNN HTP (NPU) delegate.
Completed stage: Graph Preparation Initializing (861 us)
Starting stage: Graph Optimizations
Completed stage: Graph Optimizations (1694258 us)
Starting stage: Post Graph Optimization
Completed stage: Post Graph Optimization (53009 us)
Starting stage: Graph Sequencing for Target
Completed stage: Graph Sequencing for Target (191505 us)
Starting stage: VTCM Allocation
Completed stage: VTCM Allocation (23076 us)
Starting stage: Parallelization Optimization
Completed stage: Parallelization Optimization (27541 us)
Starting stage: Finalizing Graph Sequence

====== DDR bandwidth summary ======
spill_bytes=0
fill_bytes=0
write_total_bytes=98304
read_total_bytes=29335552

Completed stage: Finalizing Graph Sequence (25375 us)
Starting stage: Completion
Completed stage: Completion (2186 us)
Active backend: npu
Input size (H,W): (256, 192)
Output quantization: scale=0.004067708272486925, zero_point=8


In [68]:
COCO_KEYPOINT_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle",
]

def get_joint_xy_conf(name, heatmaps, img_w, img_h, hm_w, hm_h):
    j = COCO_KEYPOINT_NAMES.index(name)
    jh = heatmaps[:, :, j].astype(np.float64)
    y, x = np.unravel_index(np.argmax(jh), jh.shape)
    conf = (float(jh[y, x]) - zero_point) * scale

    # Sub-pixel refinement via parabolic interpolation around the peak bin.
    # Raw argmax only has heatmap-grid precision (can be tens of pixels on a
    # high-res source frame), which was causing large spurious frame-to-frame
    # jitter in downstream angle calculations even when confidence was high.
    def _refine(center, lo, hi):
        denom = lo - 2 * center + hi
        if abs(denom) < 1e-6:
            return 0.0
        return 0.5 * (lo - hi) / denom

    dx = dy = 0.0
    if 0 < x < jh.shape[1] - 1:
        dx = _refine(jh[y, x], jh[y, x - 1], jh[y, x + 1])
    if 0 < y < jh.shape[0] - 1:
        dy = _refine(jh[y, x], jh[y - 1, x], jh[y + 1, x])

    px = (x + dx) * img_w / hm_w
    py = (y + dy) * img_h / hm_h
    return (px, py), conf

def get_joint_xy_conf_filtered(name, heatmaps, img_w, img_h, hm_w, hm_h, prev_xy=None, dt=None,
                                 max_speed_px_per_sec=4000):
    """Same as get_joint_xy_conf, but rejects a detection that implies an
    impossible jump from the previous frame (fast motion blur can make the
    model confidently lock onto background instead of the joint). Falls back
    to the raw detection with confidence zeroed out, so downstream code
    treats it as low-confidence rather than trusting a bad position."""
    xy, conf = get_joint_xy_conf(name, heatmaps, img_w, img_h, hm_w, hm_h)
    if prev_xy is not None and dt is not None and dt > 0:
        speed = np.linalg.norm(np.array(xy) - np.array(prev_xy)) / dt
        if speed > max_speed_px_per_sec:
            return xy, 0.0  # flag as untrustworthy; position kept for reference only
    return xy, conf

def joint_angle(a, b, c):
    """Angle at point b, formed by points a-b-c, in degrees."""
    a, b, c = np.array(a, dtype=np.float64), np.array(b, dtype=np.float64), np.array(c, dtype=np.float64)
    ba = a - b
    bc = c - b
    cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))

In [69]:
def extract_pose_signals(video_path):
    """Runs the full clip through the model once, returns per-frame joint positions + confidences."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)

    tracked_joints = [
        "nose",
        "left_shoulder", "right_shoulder",
        "left_elbow", "right_elbow",
        "left_wrist", "right_wrist",
        "left_hip", "right_hip",
        "left_knee", "right_knee",
        "left_ankle", "right_ankle",
    ]

    data = {"timestamps": []}
    for name in tracked_joints:
        data[f"{name}_xy"] = []
        data[f"{name}_conf"] = []

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(rgb, (IN_W, IN_H))
        model_input = np.expand_dims(resized, axis=0).astype(np.uint8)

        interpreter_npu.set_tensor(input_details[0]["index"], model_input)
        interpreter_npu.invoke()
        heatmaps = interpreter_npu.get_tensor(output_details[0]["index"])[0]

        hm_h, hm_w, _ = heatmaps.shape
        img_h, img_w = frame.shape[0], frame.shape[1]

        for name in tracked_joints:
            xy, conf = get_joint_xy_conf(name, heatmaps, img_w, img_h, hm_w, hm_h)
            data[f"{name}_xy"].append(xy)
            data[f"{name}_conf"].append(conf)

        data["timestamps"].append(frame_idx / fps)
        frame_idx += 1

    cap.release()
    for key in data:
        data[key] = np.array(data[key])
    data["fps"] = fps
    return data


In [70]:
def smooth(signal, window=3):
    if len(signal) < window:
        return signal
    return np.convolve(signal, np.ones(window)/window, mode="same")


def compute_onset_by_velocity(values, timestamps, confidences, confidence_threshold=0.4,
                                velocity_threshold_multiplier=3.0, smoothing_window=3):
    valid = confidences >= confidence_threshold
    if valid.sum() < 5:
        return None

    t = timestamps[valid]
    v = smooth(values[valid], smoothing_window)
    velocity = np.abs(np.diff(v)) / np.diff(t)
    v_t = t[1:]

    n_ref = max(3, int(0.15 * len(velocity)))
    resting_mean = velocity[:n_ref].mean()
    resting_std = velocity[:n_ref].std() + 1e-6
    threshold = resting_mean + velocity_threshold_multiplier * resting_std

    for ti, vi in zip(v_t[n_ref:], velocity[n_ref:]):
        if vi > threshold:
            return ti
    return None

'''def compute_onset_by_angle_change(angles, timestamps, confidences, angle_change_threshold=15.0,
                                    confidence_threshold=0.5, reference_frames=5):
    valid = confidences >= confidence_threshold
    if valid.sum() < reference_frames + 1:
        return None

    t = timestamps[valid]
    a = angles[valid]

    reference_angle = np.median(a[:reference_frames])  # "resting" position, from a few real early frames

    for ti, ai in zip(t, a):
        if abs(ai - reference_angle) > angle_change_threshold:
            return ti
    return None'''


def compute_settle_time_after(values, timestamps, confidences, after_time,
                                confidence_threshold=0.4, smoothing_window=3):
    valid = (confidences >= confidence_threshold) & (timestamps >= after_time)
    if valid.sum() < 5:
        return None

    t = timestamps[valid]
    v = smooth(values[valid], smoothing_window)
    velocity = np.abs(np.diff(v)) / np.diff(t)
    v_t = t[1:]

    quiet_threshold = np.percentile(velocity, 20)
    peak_idx = np.argmax(velocity)
    for i in range(peak_idx, len(velocity)):
        if velocity[i] <= quiet_threshold * 1.5:
            return v_t[i]
    return None


def body_scale(data, confidence_threshold=0.4):
    conf_mask = (data["left_shoulder_conf"] >= confidence_threshold) & \
                (data["right_shoulder_conf"] >= confidence_threshold)
    if conf_mask.sum() < 3:
        return None
    widths = np.linalg.norm(data["left_shoulder_xy"][conf_mask] - data["right_shoulder_xy"][conf_mask], axis=1)
    return np.median(widths)

In [71]:
def sequencing_score(data):
    t = data["timestamps"]

    hip_angle = np.array([
        joint_angle(data["left_hip_xy"][i], data["right_hip_xy"][i], data["right_knee_xy"][i])
        for i in range(len(t))
    ])
    hip_conf = np.minimum(np.minimum(data["left_hip_conf"], data["right_hip_conf"]), data["right_knee_conf"])

    shoulder_angle = np.array([
        joint_angle(data["left_shoulder_xy"][i], data["right_shoulder_xy"][i], data["right_elbow_xy"][i])
        for i in range(len(t))
    ])
    shoulder_conf = np.minimum(np.minimum(data["left_shoulder_conf"], data["right_shoulder_conf"]), data["right_elbow_conf"])

    hip_onset = compute_onset_by_velocity(hip_angle, t, hip_conf)
    shoulder_onset = compute_onset_by_velocity(shoulder_angle, t, shoulder_conf)

    if hip_onset is None or shoulder_onset is None:
        return {"metric": "sequencing", "message": "Not enough confident data to score.", "verdict": None}

    gap = shoulder_onset - hip_onset
    verdict = gap > 0.02
    return {
        "metric": "sequencing",
        "hip_onset": hip_onset,
        "shoulder_onset": shoulder_onset,
        "gap_seconds": gap,
        "verdict": verdict,
        "message": "Great sequencing \u2014 hips rotating before shoulders." if verdict
                   else "Throw is becoming arm-dominant \u2014 shoulders rotating too early relative to hips.",
    }


In [72]:
def front_side_stability_score(data, lead_leg="left"):
    t = data["timestamps"]
    scale_ref = body_scale(data)
    if scale_ref is None:
        return {"metric": "front_side_stability", "message": "Not enough confident data to establish body scale.", "verdict": None}

    ankle_xy = data[f"{lead_leg}_ankle_xy"]
    ankle_conf = data[f"{lead_leg}_ankle_conf"]
    knee_xy = data[f"{lead_leg}_knee_xy"]
    knee_conf = data[f"{lead_leg}_knee_conf"]

    ankle_x = ankle_xy[:, 0]
    stride_onset = compute_onset_by_velocity(ankle_x, t, ankle_conf)
    if stride_onset is None:
        return {"metric": "front_side_stability", "message": "Could not detect stride onset.", "verdict": None}

    foot_strike = compute_settle_time_after(ankle_x, t, ankle_conf, after_time=stride_onset)
    if foot_strike is None:
        return {"metric": "front_side_stability", "message": "Could not detect foot strike.", "verdict": None}

    after = (t >= foot_strike) & (knee_conf >= 0.4)
    if after.sum() < 3:
        return {"metric": "front_side_stability", "message": "Not enough confident data after foot strike.", "verdict": None}

    positions = knee_xy[after]
    drift_px = np.linalg.norm(positions - positions[0], axis=1)
    max_drift_normalized = drift_px.max() / scale_ref

    verdict = max_drift_normalized < 0.15  # placeholder - tune tomorrow
    return {
        "metric": "front_side_stability",
        "foot_strike_time": foot_strike,
        "max_drift_normalized": max_drift_normalized,
        "verdict": verdict,
        "message": "Strong brace \u2014 lead leg stabilized well." if verdict
                   else "Front side collapsing \u2014 knee drifting after plant.",
    }


In [73]:
def run_all_metrics(video_path, lead_leg="left"):
    data = extract_pose_signals(video_path)
    results = {
        "sequencing": sequencing_score(data),
        "front_side_stability": front_side_stability_score(data, lead_leg=lead_leg),
    }
    for name, r in results.items():
        print(f"--- {name} ---")
        for k, v in r.items():
            print(f"  {k}: {v}")
        print()
    return results, data


results, data = run_all_metrics("26.mp4", lead_leg="left")


--- sequencing ---
  metric: sequencing
  hip_onset: 1.4026901960784315
  shoulder_onset: 1.3024980392156864
  gap_seconds: -0.10019215686274507
  verdict: False
  message: Throw is becoming arm-dominant — shoulders rotating too early relative to hips.

--- front_side_stability ---
  metric: front_side_stability
  message: Could not detect foot strike.
  verdict: None



In [74]:
import os
import glob

def run_all_metrics_batch(folder="data", lead_leg="left", extensions=(".mp4", ".mov", ".avi", ".m4v")):
    """Runs run_all_metrics on every video in `folder`, one at a time.
    Returns {filename: (results, data)}. A video that errors out (bad file,
    model failure, etc.) is reported and skipped rather than stopping the
    whole batch."""
    ext_set = {e.lower() for e in extensions}
    video_paths = sorted(
        p for p in glob.glob(os.path.join(folder, "*"))
        if os.path.splitext(p)[1].lower() in ext_set
    )
    if not video_paths:
        print(f"No videos found in '{folder}' (looked for {extensions}).")
        return {}

    print(f"Found {len(video_paths)} video(s) in '{folder}':")
    for p in video_paths:
        print(" ", os.path.basename(p))
    print()

    all_results = {}
    for path in video_paths:
        name = os.path.basename(path)
        print("=" * 60)
        print(name)
        print("=" * 60)
        try:
            results, data = run_all_metrics(path, lead_leg=lead_leg)
            all_results[name] = (results, data)
        except Exception as e:
            print(f"  FAILED: {type(e).__name__}: {e}")
            all_results[name] = (None, None)
        print()

    print("=" * 60)
    print("SUMMARY")
    print("=" * 60)
    for name, (results, _) in all_results.items():
        if results is None:
            print(f"{name:30s}  ERROR")
            continue
        seq = results.get("sequencing", {})
        fss = results.get("front_side_stability", {})
        seq_v = seq.get("verdict")
        fss_v = fss.get("verdict")
        print(f"{name:30s}  sequencing={seq_v!s:6s}  front_side_stability={fss_v!s:6s}")

    return all_results


all_results = run_all_metrics_batch(folder="data", lead_leg="left")

Found 44 video(s) in 'data':
  1.mp4
  10.mp4
  11.mp4
  12.mp4
  13.mp4
  14.mp4
  15.mp4
  16.mp4
  17.mp4
  18.mp4
  19.mp4
  2.mp4
  20.mp4
  21.mp4
  22.mp4
  23.mp4
  24.mp4
  25.mp4
  26.mp4
  27.mp4
  28.mp4
  29.mp4
  3.mp4
  30.mp4
  31.mp4
  32.mp4
  33.mp4
  34.mp4
  35.mp4
  36.mp4
  37.mp4
  38.mp4
  39.mp4
  4.mp4
  40.mp4
  41.mp4
  42.mp4
  43.mp4
  44.mp4
  5.mp4
  6.mp4
  7.mp4
  8.mp4
  9.mp4

1.mp4
--- sequencing ---
  metric: sequencing
  hip_onset: 2.1704404958677688
  shoulder_onset: 2.4550884297520663
  gap_seconds: 0.28464793388429754
  verdict: True
  message: Great sequencing — hips rotating before shoulders.

--- front_side_stability ---
  metric: front_side_stability
  message: Could not detect foot strike.
  verdict: None


10.mp4
--- sequencing ---
  metric: sequencing
  hip_onset: 1.5947619047619046
  shoulder_onset: 1.311248677248677
  gap_seconds: -0.2835132275132275
  verdict: False
  message: Throw is becoming arm-dominant — shoulders rotating too e

In [94]:
for name, (results, data) in all_results.items():
    if results is None or data is None:
        continue
    seq = results.get("sequencing", {})
    if seq.get("verdict") is None:
        hip_conf = np.minimum(np.minimum(data["left_hip_conf"], data["right_hip_conf"]), data["right_knee_conf"])
        sh_conf = np.minimum(np.minimum(data["left_shoulder_conf"], data["right_shoulder_conf"]), data["right_elbow_conf"])
        print(f"{name}: hip_valid={sum(hip_conf>=0.4)}/{len(hip_conf)}  "
              f"shoulder_valid={sum(sh_conf>=0.4)}/{len(sh_conf)}  "
              f"msg='{seq.get('message')}'")

27.mp4: hip_valid=70/90  shoulder_valid=80/90  msg='Not enough confident data to score.'
29.mp4: hip_valid=96/96  shoulder_valid=93/96  msg='Not enough confident data to score.'
32.mp4: hip_valid=46/57  shoulder_valid=46/57  msg='Not enough confident data to score.'
34.mp4: hip_valid=63/104  shoulder_valid=62/104  msg='Not enough confident data to score.'
35.mp4: hip_valid=215/223  shoulder_valid=212/223  msg='Not enough confident data to score.'
36.mp4: hip_valid=71/90  shoulder_valid=60/90  msg='Not enough confident data to score.'
37.mp4: hip_valid=140/160  shoulder_valid=136/160  msg='Not enough confident data to score.'
42.mp4: hip_valid=91/91  shoulder_valid=84/91  msg='Not enough confident data to score.'
7.mp4: hip_valid=101/104  shoulder_valid=103/104  msg='Not enough confident data to score.'


In [64]:
def diagnose_no_onset(data):
    t = data["timestamps"]

    hip_angle = np.array([
        joint_angle(data["left_hip_xy"][i], data["right_hip_xy"][i], data["right_knee_xy"][i])
        for i in range(len(t))
    ])
    hip_conf = np.minimum(np.minimum(data["left_hip_conf"], data["right_hip_conf"]), data["right_knee_conf"])

    shoulder_angle = np.array([
        joint_angle(data["left_shoulder_xy"][i], data["right_shoulder_xy"][i], data["right_elbow_xy"][i])
        for i in range(len(t))
    ])
    shoulder_conf = np.minimum(np.minimum(data["left_shoulder_conf"], data["right_shoulder_conf"]), data["right_elbow_conf"])

    for name, values, conf in [("hip", hip_angle, hip_conf), ("shoulder", shoulder_angle, shoulder_conf)]:
        valid = conf >= 0.4
        tt = t[valid]
        v = smooth(values[valid], 3)
        velocity = np.abs(np.diff(v)) / np.diff(tt)
        n_ref = max(3, int(0.15 * len(velocity)))
        resting_mean = velocity[:n_ref].mean()
        resting_std = velocity[:n_ref].std() + 1e-6
        threshold = resting_mean + 3.0 * resting_std
        post_baseline = velocity[n_ref:]
        print(f"{name}: n_ref={n_ref}  baseline_window=({tt[0]:.2f}s to {tt[n_ref]:.2f}s)  "
              f"threshold={threshold:.2f}  max_velocity_after_baseline={post_baseline.max():.2f}  "
              f"ever_crosses={post_baseline.max() > threshold}")

diagnose_no_onset(all_results["27.mp4"][1])

hip: n_ref=10  baseline_window=(0.04s to 0.47s)  threshold=954.72  max_velocity_after_baseline=1445.20  ever_crosses=True
shoulder: n_ref=11  baseline_window=(0.04s to 0.44s)  threshold=2574.84  max_velocity_after_baseline=1261.70  ever_crosses=False


In [65]:
def peek_shoulder_baseline(data, n_ref=11):
    t = data["timestamps"]
    shoulder_angle = np.array([
        joint_angle(data["left_shoulder_xy"][i], data["right_shoulder_xy"][i], data["right_elbow_xy"][i])
        for i in range(len(t))
    ])
    shoulder_conf = np.minimum(np.minimum(data["left_shoulder_conf"], data["right_shoulder_conf"]), data["right_elbow_conf"])
    valid = shoulder_conf >= 0.4
    tt = t[valid]
    aa = shoulder_angle[valid]
    for i in range(min(n_ref + 3, len(tt))):
        print(f"frame {i}: t={tt[i]:.3f}  shoulder_angle={aa[i]:7.2f}  "
              f"right_shoulder={data['right_shoulder_xy'][valid][i]}  right_elbow={data['right_elbow_xy'][valid][i]}")

peek_shoulder_baseline(all_results["27.mp4"][1])

frame 0: t=0.036  shoulder_angle=  96.17  right_shoulder=[506.4516129  347.36758475]  right_elbow=[527.27272727 349.3125    ]
frame 1: t=0.073  shoulder_angle= 117.03  right_shoulder=[497.09090909 348.17095588]  right_elbow=[529.62962963 350.75892857]
frame 2: t=0.109  shoulder_angle= 161.46  right_shoulder=[524.     345.9375]  right_elbow=[576.92307692 347.2875    ]
frame 3: t=0.145  shoulder_angle=   2.73  right_shoulder=[501.86046512 351.46551724]  right_elbow=[746.51162791 347.66826923]
frame 4: t=0.181  shoulder_angle=   0.37  right_shoulder=[509.14285714 354.00815217]  right_elbow=[749.52380952 350.08709016]
frame 5: t=0.218  shoulder_angle=   9.25  right_shoulder=[487.14285714 370.43346774]  right_elbow=[832.83950617 387.74147727]
frame 6: t=0.254  shoulder_angle= 172.36  right_shoulder=[480.      372.09375]  right_elbow=[393.09090909 363.35685484]
frame 7: t=0.290  shoulder_angle= 175.01  right_shoulder=[510.32258065 370.9375    ]  right_elbow=[370.         356.15131579]
frame 

In [50]:
hip_angle = np.array([
    joint_angle(data["left_hip_xy"][i], data["right_hip_xy"][i], data["right_knee_xy"][i])
    for i in range(len(data["timestamps"]))
])
print("dtype:", hip_angle.dtype)
if hip_angle.dtype == object:
    bad = [i for i, v in enumerate(hip_angle) if v is None]
    print("None at indices:", bad)
    for i in bad[:5]:
        print(i, data["left_hip_xy"][i], data["right_hip_xy"][i], data["right_knee_xy"][i])
else:
    print("no None values -- array is proper float64, unrelated to hip_angle")

dtype: object
None at indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120]
0 [845.         548.87019231] [753.65853659 547.00741525] [754.07407407 740.25      ]
1 [843.07692308 547.28693182] [754.35897436 547.13942308] [753.65853659 736.47321429]
2 [844.16666667 547.2027439 ] [752.94117647 547.74872449] [756.47058824 736.54411765]
3 [843.47826087 547.2027439 ] [754.         546.36513158] [754.7826087  736.78427419]
4 [842.04081633 548.4375    ] [754.63414634 548.87019231] [758.03921569 737.09134615]


In [ ]:
def diagnose_sequencing(data):
    t = data["timestamps"]

    hip_angle = np.array([
        joint_angle(data["left_hip_xy"][i], data["right_hip_xy"][i], data["right_knee_xy"][i])
        for i in range(len(t))
    ])
    hip_conf = np.minimum(np.minimum(data["left_hip_conf"], data["right_hip_conf"]), data["right_knee_conf"])

    shoulder_angle = np.array([
        joint_angle(data["left_shoulder_xy"][i], data["right_shoulder_xy"][i], data["right_elbow_xy"][i])
        for i in range(len(t))
    ])
    shoulder_conf = np.minimum(np.minimum(data["left_shoulder_conf"], data["right_shoulder_conf"]), data["right_elbow_conf"])

    hip_valid = hip_conf >= 0.4
    sh_valid = shoulder_conf >= 0.4

    print(f"total frames: {len(t)}")
    print(f"hip valid frames: {hip_valid.sum()}   shoulder valid frames: {sh_valid.sum()}")
    print(f"valid masks identical: {np.array_equal(hip_valid, sh_valid)}")

    for name, values, conf in [("hip", hip_angle, hip_conf), ("shoulder", shoulder_angle, shoulder_conf)]:
        valid = conf >= 0.4
        if valid.sum() < 5:
            print(f"{name}: not enough valid frames ({valid.sum()})")
            continue
        tt = t[valid]
        v = smooth(values[valid], 3)
        velocity = np.abs(np.diff(v)) / np.diff(tt)
        v_t = tt[1:]
        n_ref = max(3, int(0.15 * len(velocity)))
        resting_mean = velocity[:n_ref].mean()
        resting_std = velocity[:n_ref].std() + 1e-6
        threshold = resting_mean + 3.0 * resting_std
        onset_idx = next((i for i, vi in enumerate(velocity) if i >= n_ref and vi > threshold), None)
        print(f"{name}: n_ref={n_ref}  resting_mean={resting_mean:.3f}  resting_std={resting_std:.5f}  "
              f"threshold={threshold:.3f}  onset_idx={onset_idx}  onset_time={v_t[onset_idx] if onset_idx is not None else None}")

diagnose_sequencing(data)

total frames: 121
hip valid frames: 86   shoulder valid frames: 90
valid masks identical: False
hip: n_ref=12  resting_mean=75.785  resting_std=229.33263  threshold=763.783  onset_idx=58  onset_time=2.1704404958677688
shoulder: n_ref=13  resting_mean=132.451  resting_std=246.17050  threshold=870.962  onset_idx=67  onset_time=2.4550884297520663


In [36]:
def peek_raw_angles(data, n=10):
    t = data["timestamps"]
    hip_angle = np.array([
        joint_angle(data["left_hip_xy"][i], data["right_hip_xy"][i], data["right_knee_xy"][i])
        for i in range(len(t))
    ])
    shoulder_angle = np.array([
        joint_angle(data["left_shoulder_xy"][i], data["right_shoulder_xy"][i], data["right_elbow_xy"][i])
        for i in range(len(t))
    ])
    hip_conf = np.minimum(np.minimum(data["left_hip_conf"], data["right_hip_conf"]), data["right_knee_conf"])
    shoulder_conf = np.minimum(np.minimum(data["left_shoulder_conf"], data["right_shoulder_conf"]), data["right_elbow_conf"])
    for i in range(n):
        print(f"frame {i}: t={t[i]:.3f}  hip_angle={hip_angle[i]:7.2f} (conf={hip_conf[i]:.2f})  "
              f"shoulder_angle={shoulder_angle[i]:7.2f} (conf={shoulder_conf[i]:.2f})")

peek_raw_angles(data)

frame 0: t=0.000  hip_angle=  88.71 (conf=0.73)  shoulder_angle= 104.16 (conf=0.75)
frame 1: t=0.036  hip_angle=  90.12 (conf=0.74)  shoulder_angle= 104.54 (conf=0.77)
frame 2: t=0.071  hip_angle=  89.27 (conf=0.73)  shoulder_angle= 103.98 (conf=0.81)
frame 3: t=0.107  hip_angle=  89.23 (conf=0.74)  shoulder_angle= 102.07 (conf=0.78)
frame 4: t=0.142  hip_angle=  89.25 (conf=0.72)  shoulder_angle= 102.65 (conf=0.77)
frame 5: t=0.178  hip_angle=  90.07 (conf=0.74)  shoulder_angle= 102.54 (conf=0.78)
frame 6: t=0.213  hip_angle=  90.45 (conf=0.75)  shoulder_angle= 101.11 (conf=0.76)
frame 7: t=0.249  hip_angle=  89.92 (conf=0.76)  shoulder_angle= 111.78 (conf=0.69)
frame 8: t=0.285  hip_angle=  90.10 (conf=0.76)  shoulder_angle= 114.81 (conf=0.67)
frame 9: t=0.320  hip_angle=  89.55 (conf=0.76)  shoulder_angle= 110.84 (conf=0.69)


In [25]:
def peek_raw_xy(data, n=10):
    for i in range(n):
        print(f"frame {i}: left_hip={data['left_hip_xy'][i]}  right_hip={data['right_hip_xy'][i]}  right_knee={data['right_knee_xy'][i]}")

peek_raw_xy(data)

frame 0: left_hip=[840.    556.875]  right_hip=[760. 540.]  right_knee=[760.  742.5]
frame 1: left_hip=[840. 540.]  right_hip=[760. 540.]  right_knee=[760.  742.5]
frame 2: left_hip=[840. 540.]  right_hip=[760. 540.]  right_knee=[760.  742.5]
frame 3: left_hip=[840. 540.]  right_hip=[760. 540.]  right_knee=[760.  742.5]
frame 4: left_hip=[840. 540.]  right_hip=[760.    556.875]  right_knee=[760.  742.5]
frame 5: left_hip=[840. 540.]  right_hip=[760. 540.]  right_knee=[760.    725.625]
frame 6: left_hip=[840. 540.]  right_hip=[760.    556.875]  right_knee=[760.  742.5]
frame 7: left_hip=[840. 540.]  right_hip=[760. 540.]  right_knee=[760.  742.5]
frame 8: left_hip=[840. 540.]  right_hip=[760. 540.]  right_knee=[760.  742.5]
frame 9: left_hip=[840. 540.]  right_hip=[760. 540.]  right_knee=[760.  742.5]


In [ ]:
'''hip_onset_simple = compute_onset_by_angle_change(hip_angle, data["timestamps"], hip_conf)
shoulder_onset_simple = compute_onset_by_angle_change(shoulder_angle, data["timestamps"], shoulder_conf)

print("hip_onset (simple):", hip_onset_simple)
print("shoulder_onset (simple):", shoulder_onset_simple)'''

In [ ]:
'''gap_simple = shoulder_onset_simple - hip_onset_simple
print("gap:", gap_simple)'''

In [ ]:
'''for threshold in [10, 15, 20, 25, 30]:
    h = compute_onset_by_angle_change(hip_angle, data["timestamps"], hip_conf, angle_change_threshold=threshold)
    s = compute_onset_by_angle_change(shoulder_angle, data["timestamps"], shoulder_conf, angle_change_threshold=threshold)
    if h is not None and s is not None:
        print(f"threshold={threshold:2d}°  hip_onset={h:.3f}  shoulder_onset={s:.3f}  gap={s-h:+.3f}")
    else:
        print(f"threshold={threshold:2d}°  hip_onset={h}  shoulder_onset={s}")'''

In [ ]:
'''def compute_onset_by_relative_angle_change(angles, timestamps, confidences, fraction_of_range=0.25,
                                             confidence_threshold=0.5, reference_frames=5):
    valid = confidences >= confidence_threshold
    if valid.sum() < reference_frames + 1:
        return None

    t = timestamps[valid]
    a = angles[valid]

    reference_angle = np.median(a[:reference_frames])
    total_range = np.max(np.abs(a - reference_angle))
    if total_range < 1e-6:
        return None
    threshold = fraction_of_range * total_range

    for ti, ai in zip(t, a):
        if abs(ai - reference_angle) > threshold:
            return ti
    return None

hip_onset_rel = compute_onset_by_relative_angle_change(hip_angle, data["timestamps"], hip_conf)
shoulder_onset_rel = compute_onset_by_relative_angle_change(shoulder_angle, data["timestamps"], shoulder_conf)

print("hip_onset (relative):", hip_onset_rel)
print("shoulder_onset (relative):", shoulder_onset_rel)
print("gap:", shoulder_onset_rel - hip_onset_rel if hip_onset_rel and shoulder_onset_rel else None)'''

In [ ]:
SKELETON_EDGES = [
    (5, 7), (7, 9),        # left shoulder -> elbow -> wrist
    (6, 8), (8, 10),       # right shoulder -> elbow -> wrist
    (5, 6),                 # shoulder to shoulder
    (5, 11), (6, 12),      # shoulders to hips
    (11, 12),                # hip to hip
    (11, 13), (13, 15),    # left hip -> knee -> ankle
    (12, 14), (14, 16),    # right hip -> knee -> ankle
]

CONFIDENCE_DRAW_THRESHOLD = 0.4

def draw_skeleton(display, heatmaps, img_w, img_h, hm_w, hm_h):
    """Draws every joint + bone onto `display`, color-coded by confidence."""
    positions = {}
    for j, name in enumerate(COCO_KEYPOINT_NAMES):
        (px, py), conf = get_joint_xy_conf(name, heatmaps, img_w, img_h, hm_w, hm_h)
        if conf < CONFIDENCE_DRAW_THRESHOLD:
            continue
        positions[j] = (int(px), int(py))
        cv2.circle(display, (int(px), int(py)), 5, (0, 255, 0), -1)

    for a, b in SKELETON_EDGES:
        if a in positions and b in positions:
            cv2.line(display, positions[a], positions[b], (255, 255, 0), 2)


def save_skeleton_video(input_path, output_path):
    """Runs the clip through the model frame by frame and writes a new video
    with the skeleton drawn on top - same resolution and fps as the input."""
    cap = cv2.VideoCapture(input_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (frame_w, frame_h))

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(rgb, (IN_W, IN_H))
        model_input = np.expand_dims(resized, axis=0).astype(np.uint8)

        interpreter_npu.set_tensor(input_details[0]["index"], model_input)
        interpreter_npu.invoke()
        heatmaps = interpreter_npu.get_tensor(output_details[0]["index"])[0]

        hm_h, hm_w, _ = heatmaps.shape
        img_h, img_w = frame.shape[0], frame.shape[1]

        display = frame.copy()
        draw_skeleton(display, heatmaps, img_w, img_h, hm_w, hm_h)

        writer.write(display)
        frame_idx += 1

    cap.release()
    writer.release()
    print(f"wrote {frame_idx} frames to {output_path}")


save_skeleton_video("test3.mp4", "test3_skeleton.mp4")
